In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import warnings
from astropy.io import fits
from astropy.table import Table
from astropy.time import Time
from astropy.utils.exceptions import AstropyWarning

warnings.simplefilter('ignore', category=AstropyWarning)

# --- Paths ---
data_dir   = '/home/dara/Downloads/AO_0235+164_data/'
ext_csv    = '/home/dara/Downloads/AO_0235+164.csv'
output_csv = os.path.join(data_dir, 'AO_0235+164_UVOT_Corrected.csv')

# --- Extinction values (magnitudes) ---
# V, B, U: from Schlafly & Finkbeiner (2011) via AO_0235+164.csv
# W1, M2, W2: derived from E(B-V)=0.0705, scaled by Cardelli R_lambda coefficients
ext_table = pd.read_csv(ext_csv)
def get_ext(bandpass_name):
    row = ext_table[ext_table['Bandpass'].str.strip() == bandpass_name]
    return float(row['Galactic Extinction'].iloc[0])

extinction = {
    'v':  get_ext('Landolt V'),
    'b':  get_ext('Landolt B'),
    'u':  get_ext('Landolt U'),
    'w1': 0.4514,
    'm2': 0.6132,
    'w2': 0.5716,
}
print("Extinction values (A_lambda in mag):")
for f, a in extinction.items():
    print(f"  {f.upper():3s}: {a:.4f}")

# --- Filter ID from filename ---
def filter_from_filename(fname):
    f = fname.lower()
    if   'um2' in f: return 'm2'
    elif 'uw1' in f: return 'w1'
    elif 'uw2' in f: return 'w2'
    elif 'uuu' in f: return 'u'
    elif 'ubb' in f: return 'b'
    elif 'uvv' in f: return 'v'
    return None

# --- MJD: header first, then table columns ---
SWIFT_MJD_REF = 51910.0007428

def get_mjd(hdul, table):
    for hdu in hdul:
        hdr = hdu.header
        for key in ('MJD-OBS', 'MJD_OBS'):
            if key in hdr:
                return float(hdr[key])
        if 'DATE-OBS' in hdr:
            return Time(hdr['DATE-OBS'], format='isot').mjd
    for col in ('TSTART', 'MET', 'TIME'):
        if col in table.colnames:
            val = float(table[col][0])
            if val > 0:
                return SWIFT_MJD_REF + val / 86400.0
    return None

# --- Extract, correct, and combine ---
fits_files = glob.glob(os.path.join(data_dir, '*.fits'))
print(f"\nFound {len(fits_files)} FITS files.")

rows = []
skipped = 0
non_detections = 0

for fpath in fits_files:
    fname = os.path.basename(fpath)
    filt = filter_from_filename(fname)
    if filt is None:
        skipped += 1
        continue
    try:
        with fits.open(fpath) as hdul:
            t = Table.read(fpath, hdu=1)
            mjd = get_mjd(hdul, t)
            if mjd is None:
                skipped += 1
                continue

            # Skip non-detections flagged by UVOT (MAG=99 sentinel)
            mag = float(np.ma.filled(t['MAG'][0], 99.0))
            if mag >= 99.0:
                non_detections += 1
                continue

            raw_flux     = float(t['FLUX_AA'][0])
            raw_flux_err = float(t['FLUX_AA_ERR'][0])

        A = extinction[filt]
        factor = 10 ** (A / 2.5)
        rows.append({
            'MJD':            round(mjd, 5),
            'Filter':         filt.upper(),
            'Intrinsic_Flux': raw_flux * factor,
            'Flux_Error':     raw_flux_err * factor,
        })
    except Exception as e:
        print(f"  [!] {fname}: {e}")
        skipped += 1

df_out = pd.DataFrame(rows).sort_values('MJD').reset_index(drop=True)
df_out.to_csv(output_csv, index=False)

print(f"\nDone. {len(df_out)} detections saved to:\n  {output_csv}")
print(f"Non-detections (MAG=99) excluded: {non_detections}")
print(f"Skipped (no filter/MJD): {skipped}")
print(f"\nPer-filter counts:\n{df_out['Filter'].value_counts().sort_index().to_string()}")
print(f"\nFlux ranges:")
for filt, grp in df_out.groupby('Filter'):
    print(f"  {filt}: {grp['Intrinsic_Flux'].min():.3e} — {grp['Intrinsic_Flux'].max():.3e}  erg/s/cm²/Å")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv('/home/dara/Downloads/AO_0235+164_data/AO_0235+164_UVOT_Corrected.csv')

filters = ['V', 'B', 'U', 'W1', 'M2', 'W2']
colors  = ['green', 'blue', 'cyan', 'orange', 'purple', 'red']

fig, axes = plt.subplots(len(filters), 1, figsize=(12, 15), sharex=True)

for i, ax in enumerate(axes):
    subset = df[df['Filter'] == filters[i]]
    if not subset.empty:
        ax.errorbar(x=subset['MJD'],
                    y=subset['Intrinsic_Flux'],
                    yerr=subset['Flux_Error'],
                    fmt='o', color=colors[i], ecolor='gray',
                    capsize=2, markersize=4, linestyle='None',
                    label=f'Swift {filters[i]}')
        ax.set_ylabel('Flux (erg/s/cm²/Å)')
        ax.legend(loc='upper right')
        ax.grid(True, linestyle='--', alpha=0.6)
    else:
        print(f"Warning: No data found for filter {filters[i]}")

plt.xlabel('Time (MJD)', fontsize=12)
plt.suptitle('AO 0235+164 — Swift UVOT Light Curves', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('/home/dara/Downloads/AO_0235+164_data/AO_0235+164_Multiband.jpg', dpi=150, bbox_inches='tight')
plt.show()